# Notebook 00 — Synthetic Wage Dataset Generator

This notebook **generates `wages.csv`**, the dataset used in Notebook 01.

Run this notebook first (or whenever you want to regenerate the data).

---

### Why synthetic data?

In a real study you would use the actual NLSY survey. Here we use synthetic data for one powerful reason:

> **We know the true data-generating process (DGP).**

This lets us later *evaluate* whether permutation feature importance correctly recovers the true feature ranking — something impossible with real data where the ground truth is unknown.

### The model — Mincer (1974) earnings equation

$$\ln(w_i) = \alpha + \beta_1\,\text{educ}_i + \beta_2\,\text{exper}_i + \beta_3\,\text{exper}_i^2 + \mathbf{X}_i\boldsymbol{\gamma} + \varepsilon_i, \quad \varepsilon_i \sim \mathcal{N}(0,\,\sigma^2)$$

Every coefficient below is set to a value consistent with the empirical labour-economics literature.


## 1. Imports

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import os

plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False,
                     "axes.spines.right": False, "font.size": 11})

SEED = 42
np.random.seed(SEED)
print("Ready.")


## 2. True coefficients (ground truth)

These are the **known** parameters of the DGP.
In Notebook 01 we will check how well permutation importance recovers this ranking.


In [ ]:
# ── True DGP parameters ──────────────────────
INTERCEPT = 0.50

TRUE_COEF = {
    "educ"    :  0.0900,   # +9 % per additional year of schooling  (Psacharopoulos 1994)
    "exper"   :  0.0400,   # +4 % per additional year of experience
    "expersq" : -0.0006,   # diminishing returns (Mincer 1974)
    "union"   :  0.1800,   # +18 % union wage premium              (Card 1996)
    "married" :  0.1200,   # +12 % marriage premium
    "black"   : -0.1400,   # -14 % racial wage gap
    "hisp"    : -0.0500,   # -5 %  ethnic wage gap
    "south"   : -0.0800,   # -8 %  geographic differential
    "hours"   :  0.00004,  # tiny hours effect
    "poorhlth": -0.0800,   # -8 %  health penalty
}

NOISE_STD = 0.38   # residual std in log-wage units (realistic for cross-sections)
N = 2000           # sample size

# Print a summary table
print(f"{'Feature':12s}  {'True coef':>10s}  {'|coef| rank':>11s}")
print("-" * 38)
sorted_coefs = sorted(TRUE_COEF.items(), key=lambda x: abs(x[1]), reverse=True)
for rank, (feat, coef) in enumerate(sorted_coefs, 1):
    print(f"{feat:12s}  {coef:+10.4f}  {rank:>11d}")


## 3. Generate features

Each variable is drawn from a distribution calibrated to the actual NLSY sample.
Note the intentional **correlations** between features — just like real data:

- `educ` is lower on average for `black=1` and `hisp=1` (reflecting historical inequalities)
- `expersq = exper²` is by construction perfectly correlated with `exper`

These correlations are exactly what make permutation importance tricky (see Notebook 01, Section 11).


In [ ]:
np.random.seed(SEED)

# ── Binary demographics ───────────────────────────────────────────────────
black = np.random.binomial(1, 0.12, N)   # 12 % of sample
hisp  = np.random.binomial(1, 0.08, N)   # 8 %
south = np.random.binomial(1, 0.36, N)   # 36 % live in the South

# ── Education (years) — correlated with race ─────────────────────────────
# Black and Hispanic workers have lower average schooling in this dataset,
# reflecting historical access gaps. This is NOT a causal assumption —
# it is a descriptive feature of the NLSY data we are mimicking.
educ_mean = 12.8 - 0.6 * black - 0.3 * hisp
educ = np.clip(
    np.round(educ_mean + np.random.normal(0, 2.5, N)), 0, 20
).astype(int)

# ── Work experience (years) ───────────────────────────────────────────────
exper  = np.clip(
    np.round(np.random.gamma(shape=3, scale=3.5, size=N)), 0, 35
).astype(int)
expersq = exper ** 2   # NOTE: perfectly correlated with exper by construction

# ── Labour-market variables ───────────────────────────────────────────────
union    = np.random.binomial(1, 0.22, N)
married  = np.random.binomial(1, 0.54, N)
hours    = np.clip(np.random.normal(2100, 450, N), 200, 4000).round().astype(int)
poorhlth = np.random.binomial(1, 0.05, N)

print("Feature distributions:")
print(f"  educ      mean={educ.mean():.2f}  std={educ.std():.2f}  range=[{educ.min()},{educ.max()}]")
print(f"  exper     mean={exper.mean():.2f}  std={exper.std():.2f}  range=[{exper.min()},{exper.max()}]")
print(f"  union     share={union.mean():.2%}")
print(f"  married   share={married.mean():.2%}")
print(f"  black     share={black.mean():.2%}")
print(f"  hisp      share={hisp.mean():.2%}")
print(f"  south     share={south.mean():.2%}")
print(f"  poorhlth  share={poorhlth.mean():.2%}")
print(f"  hours     mean={hours.mean():.0f}  std={hours.std():.0f}")


## 4. Generate log wage (target variable)

We now apply the Mincer equation with the true coefficients defined above,
then add Gaussian noise $\varepsilon \sim \mathcal{N}(0, 0.38^2)$.


In [ ]:
# ── Structural equation ──────────────────────
lwage = (
    INTERCEPT
    + TRUE_COEF["educ"]     * educ
    + TRUE_COEF["exper"]    * exper
    + TRUE_COEF["expersq"]  * expersq
    + TRUE_COEF["union"]    * union
    + TRUE_COEF["married"]  * married
    + TRUE_COEF["black"]    * black
    + TRUE_COEF["hisp"]     * hisp
    + TRUE_COEF["south"]    * south
    + TRUE_COEF["hours"]    * hours
    + TRUE_COEF["poorhlth"] * poorhlth
    + np.random.normal(0, NOISE_STD, N)   # irreducible noise
)

print(f"Log wage — mean: {lwage.mean():.3f}  std: {lwage.std():.3f}")
print(f"           min:  {lwage.min():.3f}  max: {lwage.max():.3f}")
print()

# Theoretical R² ceiling (no model can exceed this with these features)
signal_var = lwage.var() - NOISE_STD**2
r2_ceiling = signal_var / lwage.var()
print(f"Theoretical R² ceiling (signal / total variance): {r2_ceiling:.4f}")
print("Any model's R² on test data should be below this value.")


## 5. Assemble DataFrame and save CSV

In [ ]:
df = pd.DataFrame({
    "educ"    : educ,
    "exper"   : exper,
    "expersq" : expersq,
    "union"   : union,
    "married" : married,
    "black"   : black,
    "hisp"    : hisp,
    "south"   : south,
    "hours"   : hours,
    "poorhlth": poorhlth,
    "lwage"   : lwage.round(4),
})

OUT_PATH = os.path.join(os.path.dirname(os.path.abspath("__file__")), "wages.csv")
df.to_csv(OUT_PATH, index=False)
print(f"Saved {len(df):,} rows to: {OUT_PATH}")
print()
print(df.describe().round(3).to_string())


## 6. Validation plots

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(14, 8))
axes = axes.flatten()

# 1 — Log wage distribution
axes[0].hist(df["lwage"], bins=40, color="#2563EB", edgecolor="white")
axes[0].set(title="Distribution of log wage", xlabel="lwage")

# 2 — Wage vs education
axes[1].scatter(df["educ"], df["lwage"], alpha=0.3, s=12, color="#2563EB")
# Overlay the true linear component
x_range = np.linspace(df["educ"].min(), df["educ"].max(), 100)
y_line  = INTERCEPT + TRUE_COEF["educ"] * x_range + df["lwage"].mean() - INTERCEPT - TRUE_COEF["educ"] * df["educ"].mean()
axes[1].plot(x_range, y_line, color="red", linewidth=2, label="True slope")
axes[1].legend(fontsize=9)
axes[1].set(title=f"lwage vs educ  (β={TRUE_COEF['educ']})", xlabel="educ")

# 3 — Wage vs experience (concave)
axes[2].scatter(df["exper"], df["lwage"], alpha=0.3, s=12, color="#10B981")
axes[2].set(title=f"lwage vs exper  (β={TRUE_COEF['exper']})", xlabel="exper")

# 4 — Union premium
df.boxplot(column="lwage", by="union", ax=axes[3], grid=False)
axes[3].set(title=f"Union premium  (β={TRUE_COEF['union']})", xlabel="union (1=yes)")
plt.sca(axes[3]); plt.title(f"Union premium  (β={TRUE_COEF['union']})")

# 5 — Correlation matrix
corr = df.corr()
sns.heatmap(corr, ax=axes[4], cmap="coolwarm", center=0,
            annot=False, linewidths=0.3, cbar_kws={"shrink": 0.7})
axes[4].set_title("Correlation matrix")

# 6 — exper vs expersq (perfect collinearity demo)
axes[5].scatter(df["exper"], df["expersq"], alpha=0.4, s=12, color="#F59E0B")
axes[5].set(title=f"exper vs expersq  (r={df['exper'].corr(df['expersq']):.3f})",
            xlabel="exper", ylabel="expersq")

fig.suptitle("Dataset validation plots", fontsize=13, y=1.01)
plt.tight_layout()
plt.savefig("dataset_validation.png", dpi=120, bbox_inches="tight")
plt.show()
print("Validation plots saved.")


In [ ]:
# ── Flag high correlations ───────────────────
print("Pairwise correlations > 0.3 (absolute value):")
print("-" * 45)
corr_pairs = (corr.abs()
              .where(np.triu(np.ones(corr.shape), k=1).astype(bool))
              .stack()
              .sort_values(ascending=False))
high = corr_pairs[corr_pairs > 0.3]
if len(high):
    for (f1, f2), val in high.items():
        print(f"  {f1:10s} <-> {f2:10s}  r = {val:.3f}")
else:
    print("  None")

print()
print("Note: exper & expersq have r ~ 1.0 by construction.")
print("This is the collinearity we explore in Notebook 01 Section 11.")


## 7. OLS sanity check — can we recover the true coefficients?

Before running fancy ML models in Notebook 01, let us verify the DGP is correct
by checking whether OLS recovers the true coefficients (it should, since the
data was generated from a linear model).


In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import StandardScaler

FEATURES = list(TRUE_COEF.keys())
X = df[FEATURES].values
y = df["lwage"].values

# Unstandardised OLS to recover original scale
ols = LinearRegression().fit(X, y)

print(f"{'Feature':12s}  {'True coef':>10s}  {'OLS coef':>10s}  {'Error':>8s}")
print("-" * 46)
for feat, true_c, ols_c in zip(FEATURES, TRUE_COEF.values(), ols.coef_):
    err = ols_c - true_c
    print(f"{feat:12s}  {true_c:+10.4f}  {ols_c:+10.4f}  {err:+8.4f}")
print(f"{'Intercept':12s}  {INTERCEPT:+10.4f}  {ols.intercept_:+10.4f}  {ols.intercept_-INTERCEPT:+8.4f}")

from sklearn.metrics import r2_score
r2 = r2_score(y, ols.predict(X))
print(f"\nIn-sample R²: {r2:.4f}")
print("OLS recovers the true coefficients well — the DGP is correct.")


## 8. Summary

| What was done | Detail |
|---|---|
| Observations generated | 2,000 |
| Features | 10 (education, experience, experience², union, married, black, hisp, south, hours, poor health) |
| Target | log hourly wage (`lwage`) |
| Noise std | 0.38 (realistic for cross-sectional wage data) |
| Theoretical R² ceiling | ~0.39 (no model should exceed this) |
| Output file | `wages.csv` (same folder as this notebook) |

**You are now ready to open Notebook 01** and explore permutation feature importance.

> **Tip for instructors:** You can change `TRUE_COEF`, `NOISE_STD`, or `N` and regenerate the CSV to create different teaching scenarios — e.g. making two features equally important, or adding a useless feature.
